# PyCRMKit 1.2.0 - 06. Campaigns v1.2.0, Attribution & Linkage
Ce notebook met en pratique les fonctionnalités phares introduites dans **PyCRMKit 1.2.0 Stable** :
- **Métadonnées de Campagne & Cycle de vie** : création, activation, clôture et archivage
- **Concurrence optimiste (`expected_revision`)** : contrôle de révision immuable et sémantique PATCH
- **Adhésion directe (`Direct Members`)** : ajout unitaire ou par lot avec métadonnées de source
- **Liaison de Segments Sources** : capture stable d'audiences segmentées
- **Résolution d'Audience Dédupliquée** : calcul d'audience combinée sans doublon avec traçabilité de provenance
- **Attribution & Liens de Communication** : liaison de touchpoints sans fuite du corps des messages
- **DataOps Campagnes** : import borné (max 1 000 lignes) et export paginé avec réconciliation

## 1. Initialisation de la Suite Campagne 1.2.0

In [1]:
import sys
from pathlib import Path
from datetime import datetime, timezone, timedelta

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
src_path = str(repo_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from pycrmkit import CRM, __version__
from pycrmkit.core.references import EntityReference
from pycrmkit.campaigns import CampaignUpdate
from pycrmkit.segments import Predicate, QueryOperator
from pycrmkit.exceptions import ConflictError

from pycrmkit.communication.email.provider import EmailProviderResult, EmailDeliveryStatus

class MockEmailProvider:
    def send(self, message):
        return EmailProviderResult(
            provider="mock-email-service",
            status=EmailDeliveryStatus.ACCEPTED,
            provider_message_id=f"msg-mock-{str(message.intent_id)[:8]}",
        )

crm = CRM.memory(email_provider=MockEmailProvider()).with_context(
    actor_id="campaign-director",
    correlation_id="campaign-session-v120"
)

print(f"✅ PyCRMKit version stable qualifiée : {__version__}")

✅ PyCRMKit version stable qualifiée : 1.2.0


## 2. Création de Campagne & Contrôle de Concurrence Optimiste
Le modèle de campagne intègre une révision incrémentale. Toute mise à jour spécifiant un `expected_revision` erroné est rejetée pour garantir l'intégrité en environnement multi-utilisateurs.

In [2]:
now = datetime.now(timezone.utc)

campaign = crm.campaigns.create(
    key="summit-launch-2026",
    name="Campagne de Lancement Tech Summit 2026",
    description="Campagne multi-canal ciblant les décideurs tech européens",
    starts_at=now,
    ends_at=now + timedelta(days=45),
    metadata={"budget_allocated": 75000, "channel_mix": ["email", "linkedin", "webinar"]},
)

print(f"🚀 Campagne créée : {campaign.name}")
print(f"   Clé : {campaign.key} | Statut : {campaign.status.value}")
print(f"   Révision initiale : {campaign.revision}")

# Mise à jour avec révision attendue correcte
updated_campaign = crm.campaigns.update(
    campaign.id,
    CampaignUpdate(name="Tech Summit Europe 2026 - Official Launch"),
    expected_revision=1,
)
print(f"📝 Campagne mise à jour ! Nouvelle révision : {updated_campaign.revision}")

# Tentative de mise à jour avec révision obsolète
try:
    crm.campaigns.update(
        campaign.id,
        CampaignUpdate(name="Tentative conflictuelle"),
        expected_revision=1, # Mauvaise révision, nous sommes à 2 !
    )
    print("❌ ERREUR : Le conflit n'a pas été détecté !")
except ConflictError as exc:
    print(f"🛡️ Concurrence optimiste validée : Modification concurrente rejetée : {exc}")

🚀 Campagne créée : Campagne de Lancement Tech Summit 2026
   Clé : summit-launch-2026 | Statut : draft
   Révision initiale : 1
📝 Campagne mise à jour ! Nouvelle révision : 2
🛡️ Concurrence optimiste validée : Modification concurrente rejetée : campaign revision has changed


## 3. Membres Directs et Rattachement de Segments Sources
Une campagne combine des membres ajoutés directement et des segments de contacts prédéfinis.

In [3]:
# Création de contacts tests
c_vip = crm.contacts.create(first_name="Victor", last_name="Hugo", source="direct_invite")
c_web1 = crm.contacts.create(first_name="George", last_name="Sand", source="web_event")
c_web2 = crm.contacts.create(first_name="Gustave", last_name="Flaubert", source="web_event")

# 1. Ajout d'un membre direct
direct_member = crm.campaigns.add_member(
    campaign.id,
    EntityReference("contact", c_vip.id),
    source="vip_guest_list",
    metadata={"vip_tier": "gold"},
)
print(f"👤 Membre direct ajouté : {c_vip.display_name} (source: {direct_member.source})")

# 2. Création et rattachement d'un segment source
web_segment = crm.segments.create_dynamic(
    key="web-attendees",
    name="Participants Web Event",
    entity_kind="contact",
    query=Predicate("source", QueryOperator.EQ, "web_event"),
)

attached_source = crm.campaigns.attach_segment_source(campaign.id, web_segment.id)
print(f"📎 Segment source rattaché : {web_segment.name} ({attached_source.member_count} membre(s) capturé(s))")

👤 Membre direct ajouté : Victor Hugo (source: vip_guest_list)
📎 Segment source rattaché : Participants Web Event (2 membre(s) capturé(s))


## 4. Résolution de l'Audience Dédupliquée
Le moteur de campagne unifie les membres directs et les sources de segment pour produire l'audience dédupliquée totale.

In [4]:
audience_count = crm.campaigns.audience_count(campaign.id)
audience_items = crm.campaigns.audience(campaign.id).items

print(f"🎯 Taille de l'Audience globale dédupliquée : {audience_count} contacts")
for ref in audience_items:
    print(f"  • Audience : [{ref.kind}] {ref.id}")

# Vérification d'appartenance
is_in_campaign = crm.campaigns.contains(campaign.id, EntityReference("contact", c_vip.id))
print(f"Victor Hugo fait bien partie de l'audience ? {is_in_campaign}")

🎯 Taille de l'Audience globale dédupliquée : 3 contacts
  • Audience : [contact] 90dbfd6a-dc5f-4c85-97c0-8ac072c51dbd
  • Audience : [contact] dc539129-f71a-476f-97bf-e8c4bc05e739
  • Audience : [contact] f0435f3e-aedb-4824-894d-4e5b4b6e4e71
Victor Hugo fait bien partie de l'audience ? True


## 5. Attribution & Liaison de Communications (Link Metadata)
Conformément aux spécifications de PyCRMKit 1.2.0, les liens de communication conservent des métadonnées de touchpoint sans dupliquer ni fuiter les corps des messages.

In [5]:
from pycrmkit.communication import (
    CommunicationAddress,
    CommunicationRecipient,
    CommunicationContent,
    CommunicationChannel,
)
from pycrmkit.communication.email.provider import EmailProviderResult, EmailDeliveryStatus

# Envoi d'une communication
addr = CommunicationAddress(channel=CommunicationChannel.EMAIL, value="victor.hugo@academie.fr")
sender = CommunicationAddress(channel=CommunicationChannel.EMAIL, value="summit@pycrmkit.io")
recipients = (CommunicationRecipient(address=addr, display_name="Victor Hugo"),)
content = CommunicationContent(subject="Invitation exclusive Tech Summit 2026", text_body="Cher Victor...")

comm_record = crm.email.send(
    to=recipients,
    sender=sender,
    content=content,
    references=(EntityReference("contact", c_vip.id),),
)

# Liaison de la communication à la campagne
link = crm.campaigns.link_communication(
    campaign.id,
    communication_id=comm_record.id,
    role="invitation_email",
    metadata={"channel": "email", "batch": "wave_1"},
)

print(f"🔗 Communication liée à la campagne : ID {link.communication_id}")
print(f"   Rôle : {link.role} | Métadonnées : {link.metadata}")

# Attribution d'un résultat commercial
attr = crm.campaigns.add_attribution(
    campaign.id,
    target=EntityReference("contact", c_vip.id),
    source="vip_rsvp",
    external_ref="rsvp-ticket-001",
)
print(f"🏆 Attribution enregistrée : [{attr.target.kind}] {attr.target.id} (Source: {attr.source})")

🔗 Communication liée à la campagne : ID f17f5498-02c0-49a6-86fa-f679547b46a3
   Rôle : invitation_email | Métadonnées : {'channel': 'email', 'batch': 'wave_1'}
🏆 Attribution enregistrée : [contact] dc539129-f71a-476f-97bf-e8c4bc05e739 (Source: vip_rsvp)


## 6. DataOps Campagnes : Import Borné & Export Paginé
Démonstration de `import_campaign_members` et `export_campaign_records`.

In [6]:
import io
from pycrmkit.campaigns.data_operations import (
    export_campaign_records,
    MAX_CAMPAIGN_IMPORT_ROWS,
)

print(f"📏 Limite maximale d'import par lot autorisée : {MAX_CAMPAIGN_IMPORT_ROWS} lignes")

# Export des membres d'audience au format JSONL
export_buffer = io.StringIO()
exported_count = export_campaign_records(
    crm.campaigns,
    campaign.id,
    export_buffer,
    kind="audience",
    format="jsonl",
)
exported_jsonl = export_buffer.getvalue()

print("📄 Extrait des enregistrements d'audience exportés (JSONL) :")
for line in exported_jsonl.strip().splitlines()[:2]:
    print(f"  {line}")

📏 Limite maximale d'import par lot autorisée : 1000 lignes
📄 Extrait des enregistrements d'audience exportés (JSONL) :
  {"campaign_id":"3de30a6d-5f2b-44db-84b3-a2cafad5f962","entity_kind":"contact","entity_id":"90dbfd6a-dc5f-4c85-97c0-8ac072c51dbd"}
  {"campaign_id":"3de30a6d-5f2b-44db-84b3-a2cafad5f962","entity_kind":"contact","entity_id":"dc539129-f71a-476f-97bf-e8c4bc05e739"}


## 7. Clôture de la Campagne & Assertions Finales

In [7]:
# Activation puis complétion de la campagne
active_campaign = crm.campaigns.activate(campaign.id)
print(f"🚀 Statut après activation : {active_campaign.status.value}")

completed_campaign = crm.campaigns.complete(campaign.id)
print(f"🏁 Statut après clôture : {completed_campaign.status.value}")

assert updated_campaign.revision == 2
assert audience_count == 3
assert is_in_campaign is True
assert link.role == "invitation_email"
assert attr.source == "vip_rsvp"
assert completed_campaign.status.value == "completed"

print("✨ TOUS LES TESTS DU NOTEBOOK 06 (CAMPAIGNS V1.2.0) SONT VALIDÉS AVEC SUCCÈS !")

🚀 Statut après activation : active
🏁 Statut après clôture : completed
✨ TOUS LES TESTS DU NOTEBOOK 06 (CAMPAIGNS V1.2.0) SONT VALIDÉS AVEC SUCCÈS !
